# Âge cérébral sur IXI : effet du changement d'hôpital (ClinicaDL 2.0)

**Avant de commencer :** Exécution → Modifier le type d'exécution → **GPU T4**.

Organisation :
- les scripts viennent du dépôt GitHub [Ayoub-TAREK/ixi-brain-age](https://github.com/Ayoub-TAREK/ixi-brain-age) (`git clone` la première fois, `git pull` ensuite) ;
- les données sont téléchargées dans `/content` (rapide, mais effacé à la fin de la session) ;
- les résultats sont enregistrés sur Drive (`MyDrive/test-clinicadl/outputs/`), pour ne rien perdre si la session coupe.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi
!python --version

## 2. Connecter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Récupérer les scripts depuis GitHub

À relancer après chaque `git push` depuis le PC pour avoir la dernière version.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/test-clinicadl'
WORK = '/content/test-clinicadl'
REPO = 'https://github.com/Ayoub-TAREK/ixi-brain-age.git'

# un ancien dossier sans .git (ancienne version du notebook) empêcherait le clone
if os.path.exists(WORK) and not os.path.exists(f'{WORK}/.git'):
    !rm -rf {WORK}

if not os.path.exists(WORK):
    !git clone {REPO} {WORK}
else:
    !git -C {WORK} pull

os.makedirs(f'{WORK}/data/IXI/raw', exist_ok=True)
os.makedirs(f'{DRIVE}/outputs', exist_ok=True)
if not os.path.exists(f'{WORK}/outputs'):
    os.symlink(f'{DRIVE}/outputs', f'{WORK}/outputs')
!git -C {WORK} log --oneline -3
!ls {WORK}/scripts

## 4. Installer les dépendances

In [ ]:
!pip install -q clinicadl==2.0.0 xlrd
!python -c "import clinicadl, torch; print('ClinicaDL', clinicadl.__version__, '| CUDA', torch.cuda.is_available())"

## 5. Télécharger IXI (≈ 4,5 Go, quelques minutes sur Colab)

Source officielle : https://brain-development.org/ixi-dataset/. L'étape est sautée si les images sont déjà là.

In [ ]:
import glob
%cd {WORK}/data/IXI
URL = 'http://biomedic.doc.ic.ac.uk/brain-development/downloads/IXI'
if not os.path.exists('IXI.xls'):
    !curl -fL --retry 3 -o IXI.xls {URL}/IXI.xls
if len(glob.glob('raw/*.nii.gz')) < 500:
    !curl -fL --retry 3 -o IXI-T1.tar {URL}/IXI-T1.tar
    !tar -xf IXI-T1.tar -C raw && rm IXI-T1.tar
!ls raw | wc -l

## 6. Conversion en BIDS + fichier des âges

In [ ]:
%cd {WORK}
!python scripts/ixi_to_bids.py

## 7. Contrôle du prétraitement (une image par hôpital)

In [ ]:
!python scripts/train_ixi_site_shift.py --check
from IPython.display import Image, display
for p in sorted(glob.glob(f'{WORK}/outputs/site_shift_test-IOP/check_*_sub-*.png')):
    display(Image(p))

## 8. Entraînement : Guys + HH → test sur IOP

ClinicaDL refuse d'écraser un dossier MAPS ou un split existant : `clean_run()` les supprime avant chaque lancement (les images de contrôle sont gardées).

Commencer par `--epochs 2` pour vérifier, puis lancer le vrai entraînement.

In [ ]:
import shutil
RUN_DIR = f'{WORK}/outputs/site_shift_test-IOP'

def clean_run():
    for name in ['maps', 'split', 'train_sites.tsv', 'out_of_site.tsv']:
        path = os.path.join(RUN_DIR, name)
        if os.path.isdir(path):
            shutil.rmtree(path)
        elif os.path.exists(path):
            os.remove(path)

clean_run()
!python scripts/train_ixi_site_shift.py --test-site IOP --epochs 2 --batch-size 8

In [ ]:
clean_run()
!python scripts/train_ixi_site_shift.py --test-site IOP --epochs 50 --batch-size 8

## 9. Résultats

MAE (en années) sur le test interne (`in-site`, Guys + HH) et sur l'hôpital jamais vu (`out-of-site`, IOP).

In [ ]:
import pandas as pd
for group in ['in-site', 'out-of-site']:
    for tsv in sorted(glob.glob(f'{RUN_DIR}/maps/test/group-{group}/results/*/*/metrics/aggregated.tsv')):
        mae = pd.read_csv(tsv, sep='\t')['mae'].iloc[0]
        print(f'{group:12s} MAE = {mae:.2f} ans')